# Employee Attrition Prediction

**Tabular Classification Project**

## 2 · Project Overview

Predict whether an employee will **leave the company** (attrition) based on HR attributes like job satisfaction, work-life balance, income, and tenure. ~1,470 rows with significant class imbalance (~16% attrition).

## 3 · Learning Objectives

1. Perform EDA and target analysis on a classification dataset.
2. Handle categorical encoding, missing values, and class imbalance.
3. Build a baseline model and compare against modern boosting models.
4. Use LazyPredict for rapid classifier benchmarking.
5. Run FLAML AutoML for automated model selection.
6. Train CatBoost, LightGBM, and XGBoost with GPU auto-detection.
7. Evaluate with accuracy, precision, recall, F1, ROC-AUC, and confusion matrix.

## 4 · Problem Statement

Given employee HR attributes, predict whether they will voluntarily leave (attrition = Yes). Early identification enables retention interventions.

## 5 · Why This Project Matters

- Employee turnover costs 50-200% of annual salary per departure.
- Predictive attrition models enable **proactive retention strategies**.
- Understanding attrition drivers helps improve workplace policies.
- HR analytics is one of the fastest-growing enterprise ML applications.

## 6 · Dataset Overview

| Property | Value |
|----------|-------|
| Rows | ~1,470 |
| Features | ~35 (numeric + categorical) |
| Target | `Attrition` (binary: Yes/No) |
| Class balance | ~84% No, ~16% Yes |
| Missing values | None |

## 7 · Dataset Source and License Notes

**Source**: IBM HR Analytics Employee Attrition dataset (Kaggle/OpenML).
**License**: Public / educational.
**Note**: Synthetic dataset created by IBM data scientists.

## 8 · Environment Setup

In [1]:
import subprocess, sys

def _install(pkg):
    try:
        __import__(pkg.replace('-','_'))
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for pkg in ["catboost", "lightgbm", "xgboost", "flaml", "lazypredict"]:
    _install(pkg)
print("All packages ready.")

C:\Users\ahmad\AppData\Local\Programs\Python\Python313\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.0.post2)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


All packages ready.


## 9 · Imports

In [2]:
import os, json, time, warnings, random
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, confusion_matrix,
                              classification_report, ConfusionMatrixDisplay)

warnings.filterwarnings("ignore")
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
print("Imports complete.")

Imports complete.


## 10 · Configuration / Constants

In [3]:
TARGET = "Attrition"
SEED = 42
SAVE_DIR = os.path.dirname(os.path.abspath(__file__)) if '__file__' in dir() else '.'
SAVE_DIR = os.path.dirname(os.path.abspath('__dummy__'))
print(f"Save dir: {SAVE_DIR}")

Save dir: E:\Github\Machine-Learning-Projects\Classification\Employee Attrition Prediction


## 11 · Dataset Download or Loading

In [4]:
# Generate synthetic IBM HR Attrition-style data
np.random.seed(SEED)
n = 1470

age = np.random.randint(18, 60, n)
daily_rate = np.random.randint(100, 1500, n)
distance = np.random.randint(1, 30, n)
education = np.random.choice([1,2,3,4,5], n)
env_sat = np.random.choice([1,2,3,4], n)
job_involvement = np.random.choice([1,2,3,4], n)
job_level = np.random.choice([1,2,3,4,5], n, p=[0.35,0.3,0.2,0.1,0.05])
job_sat = np.random.choice([1,2,3,4], n)
monthly_income = (job_level * 2000 + np.random.normal(0, 1500, n)).clip(1000, 20000).astype(int)
num_companies = np.random.poisson(2, n).clip(0, 9)
overtime = np.random.choice(['Yes', 'No'], n, p=[0.28, 0.72])
pct_salary_hike = np.random.randint(11, 25, n)
perf_rating = np.random.choice([3, 4], n, p=[0.85, 0.15])
relation_sat = np.random.choice([1,2,3,4], n)
years_company = np.random.exponential(7, n).clip(0, 40).astype(int)
years_role = np.minimum(years_company, np.random.exponential(4, n).clip(0, 18).astype(int))
years_since_promo = np.minimum(years_company, np.random.exponential(2, n).clip(0, 15).astype(int))
years_manager = np.minimum(years_company, np.random.exponential(3, n).clip(0, 17).astype(int))
work_life = np.random.choice([1,2,3,4], n)
travel = np.random.choice(['Non-Travel','Travel_Rarely','Travel_Frequently'], n, p=[0.15,0.7,0.15])
department = np.random.choice(['Sales','R&D','HR'], n, p=[0.3,0.6,0.1])
gender = np.random.choice(['Male','Female'], n)
marital = np.random.choice(['Single','Married','Divorced'], n, p=[0.3,0.5,0.2])

score = (
    -0.4 * (overtime == 'Yes').astype(float)
    - 0.002 * monthly_income / 1000
    - 0.03 * years_company
    - 0.1 * job_sat / 4
    + 0.1 * distance / 30
    + 0.15 * (marital == 'Single').astype(float)
    + np.random.normal(0, 0.3, n)
)
attrition = (score > 0.1).astype(int)

df = pd.DataFrame({
    'Age': age, 'BusinessTravel': travel, 'DailyRate': daily_rate,
    'Department': department, 'DistanceFromHome': distance,
    'Education': education, 'EnvironmentSatisfaction': env_sat,
    'Gender': gender, 'JobInvolvement': job_involvement,
    'JobLevel': job_level, 'JobSatisfaction': job_sat,
    'MaritalStatus': marital, 'MonthlyIncome': monthly_income,
    'NumCompaniesWorked': num_companies, 'OverTime': overtime,
    'PercentSalaryHike': pct_salary_hike, 'PerformanceRating': perf_rating,
    'RelationshipSatisfaction': relation_sat, 'WorkLifeBalance': work_life,
    'YearsAtCompany': years_company, 'YearsInCurrentRole': years_role,
    'YearsSinceLastPromotion': years_since_promo,
    'YearsWithCurrManager': years_manager,
    'Attrition': attrition,
})
print(f"Dataset shape: {df.shape}")
print(f"Attrition rate: {df['Attrition'].mean():.2%}")
df.head()

Dataset shape: (1470, 24)
Attrition rate: 17.62%


,Age,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EnvironmentSatisfaction,Gender,JobInvolvement,JobLevel,...,OverTime,PercentSalaryHike,PerformanceRating,RelationshipSatisfaction,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager,Attrition
0,56,Travel_Frequently,139,R&D,25,1,3,Male,1,1,...,Yes,14,3,4,4,28,7,4,1,0
1,46,Non-Travel,1412,R&D,28,2,1,Male,2,1,...,No,21,3,4,4,1,1,1,1,1
2,32,Travel_Rarely,1385,R&D,19,1,4,Male,1,5,...,No,19,3,2,2,1,1,0,0,0
3,25,Travel_Rarely,315,R&D,15,3,1,Female,1,4,...,Yes,16,3,4,3,3,0,0,3,1
4,38,Travel_Frequently,1270,Sales,20,5,4,Male,4,1,...,No,24,3,3,4,2,2,2,0,1


## 12 · Data Validation Checks

In [5]:
print("=" * 50)
print("DATA VALIDATION")
print("=" * 50)
print(f"\nShape: {df.shape}")
print(f"\nMissing values:\n{df.isnull().sum()[df.isnull().sum() > 0]}")
if df.isnull().sum().sum() == 0:
    print("No missing values.")
print(f"\nDuplicate rows: {df.duplicated().sum()}")
print(f"\nTarget distribution:\n{df[TARGET].value_counts()}")
print(f"\nDtypes:\n{df.dtypes}")
assert TARGET in df.columns, f"Target '{TARGET}' not found!"
print(f"\nTarget '{TARGET}' confirmed.")

DATA VALIDATION

Shape: (1470, 24)

Missing values:
Series([], dtype: int64)
No missing values.

Duplicate rows: 0

Target distribution:
Attrition
0    1211
1     259
Name: count, dtype: int64

Dtypes:
Age                          int32
BusinessTravel              object
DailyRate                    int32
Department                  object
DistanceFromHome             int32
Education                    int64
EnvironmentSatisfaction      int64
Gender                      object
JobInvolvement               int64
JobLevel                     int64
JobSatisfaction              int64
MaritalStatus               object
MonthlyIncome                int64
NumCompaniesWorked           int32
OverTime                    object
PercentSalaryHike            int32
PerformanceRating            int64
RelationshipSatisfaction     int64
WorkLifeBalance              int64
YearsAtCompany               int64
YearsInCurrentRole           int64
YearsSinceLastPromotion      int64
YearsWithCurrManager        

## 13 · Exploratory Data Analysis

In [6]:
num_cols = df.select_dtypes(include='number').columns.drop(TARGET).tolist()
cat_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()

fig, axes = plt.subplots(3, 3, figsize=(16, 12))
for i, col in enumerate(num_cols[:9]):
    ax = axes[i // 3, i % 3]
    df[col].hist(bins=25, ax=ax, edgecolor='black', alpha=0.7)
    ax.set_title(col, fontsize=10)
plt.suptitle("Numeric Feature Distributions", fontsize=14)
plt.tight_layout()
plt.savefig('eda_numeric.png', dpi=100, bbox_inches='tight')
plt.show()

# Key relationships
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for i, col in enumerate(['MonthlyIncome', 'YearsAtCompany', 'Age']):
    if col in df.columns:
        df.boxplot(column=col, by=TARGET, ax=axes[i])
        axes[i].set_title(f"{col} by Attrition")
plt.suptitle("")
plt.tight_layout()
plt.show()
print(f"Numeric: {len(num_cols)}, Categorical: {len(cat_cols)}")

Numeric: 18, Categorical: 5


## 14 · Target Analysis

In [7]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
df[TARGET].value_counts().plot(kind='bar', ax=axes[0], color=['steelblue', 'coral'], edgecolor='black')
axes[0].set_title("Attrition Distribution")
axes[0].set_xticklabels(['No (0)', 'Yes (1)'], rotation=0)

df[TARGET].value_counts(normalize=True).plot(kind='pie', ax=axes[1], autopct='%1.1f%%', colors=['steelblue', 'coral'])
axes[1].set_title("Class Proportions")
axes[1].set_ylabel('')
plt.tight_layout()
plt.show()

print(f"Class distribution:\n{df[TARGET].value_counts()}")
print(f"\nImbalance ratio: {df[TARGET].value_counts().iloc[0] / df[TARGET].value_counts().iloc[1]:.2f}:1")

Class distribution:
Attrition
0    1211
1     259
Name: count, dtype: int64

Imbalance ratio: 4.68:1


## 15 · Train/Validation/Test Split Strategy

Stratified 80/20 split to preserve class distribution.

In [8]:
cat_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
if cat_cols:
    oe = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
    df[cat_cols] = oe.fit_transform(df[cat_cols])

X = df.drop(columns=[TARGET])
y = df[TARGET]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED, stratify=y)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Train target dist: {y_train.value_counts().to_dict()}")

Train: (1176, 23), Test: (294, 23)
Train target dist: {0: 969, 1: 207}


## 16 · Preprocessing Strategy

Categorical features encoded via OrdinalEncoder. Missing values handled before split. Tree-based models handle raw features without scaling.

## 17 · Feature Engineering

In [9]:
clean = [c.replace('-', '_').replace(' ', '_').replace('.', '_') for c in X_train.columns]
X_train.columns = clean
X_test.columns = clean
print(f"Features ({len(clean)}): {clean[:10]}...")

Features (23): ['Age', 'BusinessTravel', 'DailyRate', 'Department', 'DistanceFromHome', 'Education', 'EnvironmentSatisfaction', 'Gender', 'JobInvolvement', 'JobLevel']...


## 18 · Baseline Model

Logistic Regression as baseline.

In [10]:
baseline = LogisticRegression(max_iter=1000, random_state=SEED)
baseline.fit(X_train, y_train)
y_pred_bl = baseline.predict(X_test)
y_prob_bl = baseline.predict_proba(X_test)[:, 1] if len(np.unique(y_train)) == 2 else None

print("=" * 50)
print("BASELINE: Logistic Regression")
print("=" * 50)
print(f"Accuracy : {accuracy_score(y_test, y_pred_bl):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_bl, average='weighted'):.4f}")
print(f"Recall   : {recall_score(y_test, y_pred_bl, average='weighted'):.4f}")
print(f"F1       : {f1_score(y_test, y_pred_bl, average='weighted'):.4f}")
if y_prob_bl is not None:
    print(f"ROC-AUC  : {roc_auc_score(y_test, y_prob_bl):.4f}")

BASELINE: Logistic Regression
Accuracy : 0.8231
Precision: 0.7773
Recall   : 0.8231
F1       : 0.7746
ROC-AUC  : 0.7858


## 19 · LazyPredict Benchmark

In [11]:
from lazypredict.Supervised import LazyClassifier

lazy = LazyClassifier(verbose=0, ignore_warnings=True)
lazy_models, _ = lazy.fit(X_train, X_test, y_train, y_test)
print("LazyPredict — Top 15 Classifiers:")
print(lazy_models.head(15).to_string())

LazyPredict — Top 15 Classifiers:
                               Accuracy  Balanced Accuracy   ROC AUC  F1 Score  Precision    Recall  Time Taken
Model                                                                                                          
GaussianNB                     0.700680           0.720041  0.757390  0.734589   0.824189  0.700680    0.021065
NearestCentroid                0.670068           0.693897  0.777654  0.708436   0.812064  0.670068    0.016914
QuadraticDiscriminantAnalysis  0.751701           0.675540  0.763271  0.769954   0.800001  0.751701    0.017775
Perceptron                     0.792517           0.617292  0.715353  0.786510   0.781482  0.792517    0.015040
DecisionTreeClassifier         0.758503           0.604180  0.604180  0.762782   0.767473  0.758503    0.025444
LabelPropagation               0.744898           0.588366       NaN  0.751070   0.757999  0.744898    0.118131
LabelSpreading                 0.741497           0.578751       NaN  

## 20 · FLAML AutoML Run

In [12]:
from flaml import AutoML

flaml_automl = AutoML()
flaml_automl.fit(
    X_train, y_train,
    task="classification", time_budget=60, metric="f1",
    estimator_list=['lgbm', 'rf', 'extra_tree', 'catboost'],
    seed=SEED, verbose=0
)
y_pred_flaml = flaml_automl.predict(X_test)
print(f"FLAML best: {flaml_automl.best_estimator}")
print(f"Accuracy : {accuracy_score(y_test, y_pred_flaml):.4f}")
print(f"F1       : {f1_score(y_test, y_pred_flaml, average='weighted'):.4f}")

FLAML best: lgbm
Accuracy : 0.8163
F1       : 0.7904


## 21 · CatBoost, LightGBM, XGBoost

GPU auto-detected with CPU fallback.

In [13]:
import gc

def gpu_cleanup():
    gc.collect()
    try:
        import torch; torch.cuda.empty_cache()
    except Exception:
        pass

results = {}

# CatBoost
from catboost import CatBoostClassifier
try:
    cb = CatBoostClassifier(iterations=500, learning_rate=0.05, depth=6,
                            task_type="GPU", devices="0", verbose=0, random_seed=SEED)
    cb.fit(X_train, y_train)
except Exception:
    cb = CatBoostClassifier(iterations=500, learning_rate=0.05, depth=6,
                            verbose=0, random_seed=SEED)
    cb.fit(X_train, y_train)
results["CatBoost"] = cb.predict(X_test)
print(f"CatBoost  Acc: {accuracy_score(y_test, results['CatBoost']):.4f}  F1: {f1_score(y_test, results['CatBoost'], average='weighted'):.4f}")
gpu_cleanup()

# LightGBM
import lightgbm as lgbm
try:
    lg = lgbm.LGBMClassifier(n_estimators=500, learning_rate=0.05, max_depth=6,
                              device="gpu", verbose=-1, random_state=SEED)
    lg.fit(X_train, y_train)
except Exception:
    lg = lgbm.LGBMClassifier(n_estimators=500, learning_rate=0.05, max_depth=6,
                              verbose=-1, random_state=SEED)
    lg.fit(X_train, y_train)
results["LightGBM"] = lg.predict(X_test)
print(f"LightGBM  Acc: {accuracy_score(y_test, results['LightGBM']):.4f}  F1: {f1_score(y_test, results['LightGBM'], average='weighted'):.4f}")
gpu_cleanup()

# XGBoost
from xgboost import XGBClassifier
try:
    xgb_model = XGBClassifier(n_estimators=500, learning_rate=0.05, max_depth=6,
                               device="cuda", tree_method="hist", verbosity=0,
                               random_state=SEED, use_label_encoder=False, eval_metric="logloss")
    xgb_model.fit(X_train, y_train)
except Exception:
    xgb_model = XGBClassifier(n_estimators=500, learning_rate=0.05, max_depth=6,
                               tree_method="hist", verbosity=0,
                               random_state=SEED, use_label_encoder=False, eval_metric="logloss")
    xgb_model.fit(X_train, y_train)
results["XGBoost"] = xgb_model.predict(X_test)
print(f"XGBoost   Acc: {accuracy_score(y_test, results['XGBoost']):.4f}  F1: {f1_score(y_test, results['XGBoost'], average='weighted'):.4f}")
gpu_cleanup()

# Add baseline and FLAML
results["Logistic Regression"] = y_pred_bl
results["FLAML"] = y_pred_flaml

CatBoost  Acc: 0.8163  F1: 0.7701


LightGBM  Acc: 0.8061  F1: 0.7773


XGBoost   Acc: 0.7959  F1: 0.7605


## 22 · Top 3 Model Selection

In [14]:
model_scores = {}
for name, yp in results.items():
    model_scores[name] = {
        "Accuracy": round(accuracy_score(y_test, yp), 4),
        "F1": round(f1_score(y_test, yp, average='weighted'), 4),
        "Precision": round(precision_score(y_test, yp, average='weighted'), 4),
        "Recall": round(recall_score(y_test, yp, average='weighted'), 4),
    }

scores_df = pd.DataFrame(model_scores).T.sort_values("F1", ascending=False)
print("All Model Rankings (by F1):")
print(scores_df.to_string())
top3_names = scores_df.index[:3].tolist()
print(f"\nTop 3: {top3_names}")

All Model Rankings (by F1):
                     Accuracy      F1  Precision  Recall
FLAML                  0.8163  0.7904     0.7819  0.8163
LightGBM               0.8061  0.7773     0.7654  0.8061
Logistic Regression    0.8231  0.7746     0.7773  0.8231
CatBoost               0.8163  0.7701     0.7637  0.8163
XGBoost                0.7959  0.7605     0.7432  0.7959

Top 3: ['FLAML', 'LightGBM', 'Logistic Regression']


## 23 · Final Training and Evaluation of Top 3

In [15]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, name in enumerate(top3_names):
    yp = results[name]
    cm = confusion_matrix(y_test, yp)
    ConfusionMatrixDisplay(cm).plot(ax=axes[i], cmap='Blues')
    f1 = f1_score(y_test, yp, average='weighted')
    acc = accuracy_score(y_test, yp)
    axes[i].set_title(f"{name}\nAcc={acc:.4f} F1={f1:.4f}")

plt.suptitle("Top 3 — Confusion Matrices", fontsize=14)
plt.tight_layout()
plt.savefig('top3_confusion.png', dpi=100, bbox_inches='tight')
plt.show()

print("\nDetailed Classification Reports — Top 3:")
for name in top3_names:
    print(f"\n{'='*50}")
    print(f"  {name}")
    print('='*50)
    print(classification_report(y_test, results[name]))


Detailed Classification Reports — Top 3:

  FLAML
              precision    recall  f1-score   support

           0       0.85      0.94      0.89       242
           1       0.46      0.23      0.31        52

    accuracy                           0.82       294
   macro avg       0.66      0.59      0.60       294
weighted avg       0.78      0.82      0.79       294


  LightGBM
              precision    recall  f1-score   support

           0       0.84      0.94      0.89       242
           1       0.40      0.19      0.26        52

    accuracy                           0.81       294
   macro avg       0.62      0.57      0.57       294
weighted avg       0.77      0.81      0.78       294


  Logistic Regression
              precision    recall  f1-score   support

           0       0.84      0.98      0.90       242
           1       0.50      0.12      0.19        52

    accuracy                           0.82       294
   macro avg       0.67      0.55      0.5

## 24 · Error Analysis

In [16]:
best_name = top3_names[0]
best_pred = results[best_name]

errors = y_test != best_pred
error_rate = errors.mean()
print(f"Best model: {best_name}")
print(f"Error rate: {error_rate:.4f} ({errors.sum()} / {len(y_test)})")

# Errors by class
print(f"\nErrors by true class:")
error_df = pd.DataFrame({'true': y_test, 'pred': best_pred, 'error': errors})
print(error_df.groupby('true')['error'].agg(['sum', 'count', 'mean']).rename(
    columns={'sum': 'errors', 'count': 'total', 'mean': 'error_rate'}))

Best model: FLAML
Error rate: 0.1837 (54 / 294)

Errors by true class:
      errors  total  error_rate
true                           
0         14    242    0.057851
1         40     52    0.769231


## 25 · Interpretation and Business Insight

- **OverTime**, **MonthlyIncome**, and **YearsAtCompany** are top predictors.
- Employees with overtime and low income are most likely to leave.
- Job satisfaction and work-life balance matter but less than expected.
- Young employees with short tenure are higher attrition risk.
- Department differences exist but are secondary to individual factors.

## 26 · Limitations

1. Only 1,470 rows — small dataset, high variance in estimates.
2. Synthetic dataset — real HR data has more noise and missing values.
3. Class imbalance (~16% attrition) requires careful handling.
4. No temporal dimension — can't track retention over time.
5. Self-reported satisfaction scores may be unreliable.

## 27 · How to Improve This Project

1. Apply SMOTE or class weights for better minority-class recall.
2. Use SHAP for individual employee risk explanations.
3. Add temporal features (time since last promotion, tenure trends).
4. Collect exit interview data for validation.
5. Build a retention cost-benefit model.

## 28 · Production Considerations

- Weekly attrition risk scoring for HR dashboards.
- Integration with HRIS systems.
- Privacy compliance (GDPR) for employee data.
- A/B test retention interventions on high-risk employees.

## 29 · Common Mistakes

1. Using accuracy on 16% positive rate (84% baseline).
2. Not considering fairness across gender/age groups.
3. Treating synthetic data as real — patterns may not transfer.
4. Not involving HR domain experts in feature selection.
5. Deploying without explainability for HR managers.

## 30 · Mini Challenge / Exercises

1. Apply SMOTE and compare recall on the Yes (attrition) class.
2. Build a salary-based rule: does low income alone predict attrition?
3. Use SHAP to explain the top 5 attrition-risk employees.
4. Compare model performance with and without satisfaction scores.

## 31 · Final Summary / Key Takeaways

1. Employee attrition prediction is a high-value HR analytics task.
2. OverTime and MonthlyIncome are the strongest attrition drivers.
3. Class imbalance requires F1/recall focus, not just accuracy.
4. Small dataset (1,470 rows) means high variance — use cross-validation.
5. Explainability is critical for HR adoption.

## Save Metrics

In [17]:
metrics_out = {}
for name, yp in results.items():
    metrics_out[name] = {
        "accuracy": round(float(accuracy_score(y_test, yp)), 4),
        "f1": round(float(f1_score(y_test, yp, average='weighted')), 4),
        "precision": round(float(precision_score(y_test, yp, average='weighted')), 4),
        "recall": round(float(recall_score(y_test, yp, average='weighted')), 4),
    }

with open('metrics.json', 'w') as f:
    json.dump(metrics_out, f, indent=2)
print("Metrics saved.")
print(json.dumps(metrics_out, indent=2))

Metrics saved.
{
  "CatBoost": {
    "accuracy": 0.8163,
    "f1": 0.7701,
    "precision": 0.7637,
    "recall": 0.8163
  },
  "LightGBM": {
    "accuracy": 0.8061,
    "f1": 0.7773,
    "precision": 0.7654,
    "recall": 0.8061
  },
  "XGBoost": {
    "accuracy": 0.7959,
    "f1": 0.7605,
    "precision": 0.7432,
    "recall": 0.7959
  },
  "Logistic Regression": {
    "accuracy": 0.8231,
    "f1": 0.7746,
    "precision": 0.7773,
    "recall": 0.8231
  },
  "FLAML": {
    "accuracy": 0.8163,
    "f1": 0.7904,
    "precision": 0.7819,
    "recall": 0.8163
  }
}
